In [ ]:
import numpy as np
import pandas as pd
import sys
import os
sys.path.append(os.path.abspath('..'))
import matplotlib.pyplot as plt
from scripts import nodes as n
from scripts import elements as e
from scripts import material_params as mat
from scipy.linalg import eigh
import plotly.graph_objects as go
from scripts import FDD as fdd
import pandas as pd
from scipy.optimize import minimize

## **Loading flipped all modes to compare the same modes with eachother**

In [ ]:
param_space2D_fixed = np.load('../05model_updating/param_spaces/param_space_eigvals2D_fixed.npy')
eigvecs2D_fixed = np.load('../05model_updating/param_spaces/param_space_eigvecs2D_fixed.npy')

param_space3D_fixed = np.load('../05model_updating/param_spaces/param_space_eigvals3D_fixed.npy')
eigvecs3D_fixed = np.load('../05model_updating/param_spaces/param_space_eigvecs3D_fixed.npy')

param_space3D_2_fixed = np.load('../05model_updating/param_spaces/param_space_eigvals3D_2_fixed.npy')
eigvecs3D_2_fixed = np.load('../05model_updating/param_spaces/param_space_eigvecs3D_2_fixed.npy')

## **Using four identified modes and three parameters** 

In [ ]:
N = param_space3D_fixed.shape[0]
DOF = eigvecs3D_fixed.shape[3]

kf_min, kf_max = 5e6, 5e7
Iy_min, Iy_max = 100, 2000
kl_min, kl_max = 0.2e10, 1e10

kf_vals = np.linspace(kf_min, kf_max, N)
Iy_vals = np.linspace(Iy_min, Iy_max, N)
kl_vals = np.linspace(kl_min, kl_max, N)

def params_from_indices(i, j, k):
    return Iy_vals[i], kf_vals[j], kl_vals[k]

def objective(theta, f_meas, modes_meas):
    Iy, kf, kl = theta

    i = np.argmin(abs(Iy_vals - Iy))
    j = np.argmin(abs(kf_vals - kf))
    k = np.argmin(abs(kl_vals - kl))

    f_pred = param_space3D_fixed[i, j, k, :]
    modes_pred = eigvecs3D_fixed[i, j, k, :, :]

    freq_term = np.sum(((f_pred - f_meas) / f_meas) ** 2)

    mode_term = 0
    for r in range(4):
        phi_p = modes_pred[:, r]
        phi_m = modes_meas[:, r]

        gamma = (phi_p @ phi_m) / (phi_m @ phi_m)

        mode_term += np.linalg.norm(phi_p - gamma * phi_m) ** 2 / np.linalg.norm(gamma * phi_m) ** 2

    return freq_term + mode_term

num_samples = 1000
num_inits = 50

i_rand = np.random.randint(0, N, num_samples)
j_rand = np.random.randint(0, N, num_samples)
k_rand = np.random.randint(0, N, num_samples)

results = []

tol = 0.1   

for s in range(num_samples):
    i, j, k = i_rand[s], j_rand[s], k_rand[s]

    f_meas = param_space3D_fixed[i, j, k, :]
    modes_meas = eigvecs3D_fixed[i, j, k, :, :]

    Iy_true, kf_true, kl_true = params_from_indices(i, j, k)

    success_count = 0
    best_error = np.inf
    best_solution = None

    for _ in range(num_inits):

        Iy0 = np.random.uniform(Iy_min, Iy_max)
        kf0 = np.random.uniform(kf_min, kf_max)
        kl0 = np.random.uniform(kl_min, kl_max)

        res = minimize(
            objective,
            np.array([Iy0, kf0, kl0]),
            args=(f_meas, modes_meas),
            method="Nelder-Mead",
            options={"maxiter": 300, "disp": False}
        )

        Iy_id, kf_id, kl_id = res.x

        err_Iy = abs(Iy_id - Iy_true) / Iy_true
        err_kf = abs(kf_id - kf_true) / kf_true
        err_kl = abs(kl_id - kl_true) / kl_true

        success = (err_Iy <= tol) and (err_kf <= tol) and (err_kl <= tol)

        success_count += success

        # track best solution
        total_error = (err_Iy + err_kf + err_kl) / 3
        if total_error < best_error:
            best_error = total_error
            best_solution = (Iy_id, kf_id, kl_id)

    results.append({
        "i": i,
        "j": j,
        "k": k,

        "Iy_true": Iy_true,
        "kf_true": kf_true,
        "kl_true": kl_true,

        "success_count": success_count,
        "success_rate": success_count / num_inits,

        "best_error": best_error,

        "Iy_best": best_solution[0],
        "kf_best": best_solution[1],
        "kl_best": best_solution[2],
    })

    print(f"Sample {s+1}/{num_samples} | success rate: {success_count}/{num_inits}")

df_results = pd.DataFrame(results)
df_results.to_csv("csv_files\\fem_inversion_success_3D_43.csv", index=False)

print("DONE — Results saved to fem_inversion_success.csv")

# **Using three identified modes and three parameters**

In [ ]:
N = param_space3D_fixed.shape[0]
DOF = eigvecs3D_fixed.shape[3]

kf_min, kf_max = 5e6, 5e7
Iy_min, Iy_max = 100, 2000
kl_min, kl_max = 0.2e10, 1e10

kf_vals = np.linspace(kf_min, kf_max, N)
Iy_vals = np.linspace(Iy_min, Iy_max, N)
kl_vals = np.linspace(kl_min, kl_max, N)

def params_from_indices(i, j, k):
    return Iy_vals[i], kf_vals[j], kl_vals[k]

def objective(theta, f_meas, modes_meas):
    Iy, kf, kl = theta

    i = np.argmin(abs(Iy_vals - Iy))
    j = np.argmin(abs(kf_vals - kf))
    k = np.argmin(abs(kl_vals - kl))

    f_pred = param_space3D_fixed[i, j, k, :]
    modes_pred = eigvecs3D_fixed[i, j, k, :, :]

    freq_term = np.sum(((f_pred - f_meas) / f_meas) ** 2)

    mode_term = 0
    for r in range(3):
        phi_p = modes_pred[:, r]
        phi_m = modes_meas[:, r]

        gamma = (phi_p @ phi_m) / (phi_m @ phi_m)

        mode_term += np.linalg.norm(phi_p - gamma * phi_m) ** 2 / np.linalg.norm(gamma * phi_m) ** 2

    return freq_term + mode_term

num_samples = 1000
num_inits = 50

i_rand = np.random.randint(0, N, num_samples)
j_rand = np.random.randint(0, N, num_samples)
k_rand = np.random.randint(0, N, num_samples)

results = []

tol = 0.1  

for s in range(num_samples):
    i, j, k = i_rand[s], j_rand[s], k_rand[s]

    f_meas = param_space3D_fixed[i, j, k, :]
    modes_meas = eigvecs3D_fixed[i, j, k, :, :]

    Iy_true, kf_true, kl_true = params_from_indices(i, j, k)

    success_count = 0
    best_error = np.inf
    best_solution = None

    for _ in range(num_inits):

        Iy0 = np.random.uniform(Iy_min, Iy_max)
        kf0 = np.random.uniform(kf_min, kf_max)
        kl0 = np.random.uniform(kl_min, kl_max)

        res = minimize(
            objective,
            np.array([Iy0, kf0, kl0]),
            args=(f_meas, modes_meas),
            method="Nelder-Mead",
            options={"maxiter": 300, "disp": False}
        )

        Iy_id, kf_id, kl_id = res.x

        err_Iy = abs(Iy_id - Iy_true) / Iy_true
        err_kf = abs(kf_id - kf_true) / kf_true
        err_kl = abs(kl_id - kl_true) / kl_true

        success = (err_Iy <= tol) and (err_kf <= tol) and (err_kl <= tol)

        success_count += success

        # track best solution
        total_error = (err_Iy + err_kf + err_kl) / 3
        if total_error < best_error:
            best_error = total_error
            best_solution = (Iy_id, kf_id, kl_id)

    results.append({
        "i": i,
        "j": j,
        "k": k,

        "Iy_true": Iy_true,
        "kf_true": kf_true,
        "kl_true": kl_true,

        "success_count": success_count,
        "success_rate": success_count / num_inits,

        "best_error": best_error,

        "Iy_best": best_solution[0],
        "kf_best": best_solution[1],
        "kl_best": best_solution[2],
    })

    print(f"Sample {s+1}/{num_samples} | success rate: {success_count}/{num_inits}")

df_results = pd.DataFrame(results)
df_results.to_csv("csv_files\\fem_inversion_success_3D_33.csv", index=False)

print("DONE — Results saved to fem_inversion_success.csv")

## **Using four identified modes and two parameters**

In [ ]:
N = param_space2D_fixed.shape[0]
DOF = eigvecs2D_fixed.shape[3]

kf_min, kf_max = 5e6, 5e7
Iy_min, Iy_max = 100, 2000

kf_vals = np.linspace(kf_min, kf_max, N)
Iy_vals = np.linspace(Iy_min, Iy_max, N)

def params_from_indices(i, j):
    return Iy_vals[i], kf_vals[j]

def objective(theta, f_meas, modes_meas):
    Iy, kf = theta

    i = np.argmin(np.abs(Iy_vals - Iy))
    j = np.argmin(np.abs(kf_vals - kf))

    f_pred = param_space2D_fixed[i, j, :]
    modes_pred = eigvecs2D_fixed[i, j, :, :]

    # frequency mismatch
    freq_term = np.sum(((f_pred - f_meas) / f_meas) ** 2)

    # mode shape mismatch
    mode_term = 0.0

    for r in range(4):
        phi_p = modes_pred[:, r]
        phi_m = modes_meas[:, r]

        gamma = (phi_p @ phi_m) / (phi_m @ phi_m)

        mode_term += (
            np.linalg.norm(phi_p - gamma * phi_m) ** 2
            / np.linalg.norm(gamma * phi_m) ** 2
        )

    return freq_term + mode_term

num_samples = 1000
num_inits = 50

i_rand = np.random.randint(0, N, num_samples)
j_rand = np.random.randint(0, N, num_samples)

results = []


tol = 0.10 

for s in range(num_samples):

    i = i_rand[s]
    j = j_rand[s]

    # "measured" data from the parameter space
    f_meas = param_space2D_fixed[i, j, :]
    modes_meas = eigvecs2D_fixed[i, j, :, :]

    Iy_true, kf_true = params_from_indices(i, j)

    success_count = 0
    best_error = np.inf
    best_solution = None

    for _ in range(num_inits):

        Iy0 = np.random.uniform(Iy_min, Iy_max)
        kf0 = np.random.uniform(kf_min, kf_max)

        res = minimize(
            objective,
            np.array([Iy0, kf0]),
            args=(f_meas, modes_meas),
            method="Nelder-Mead",
            options={
                "maxiter": 300,
                "disp": False
            }
        )

        Iy_id, kf_id = res.x

        err_Iy = abs(Iy_id - Iy_true) / Iy_true
        err_kf = abs(kf_id - kf_true) / kf_true

        success = (err_Iy <= tol) and (err_kf <= tol)

        success_count += success

        # track best solution
        total_error = (err_Iy + err_kf) / 2

        if total_error < best_error:
            best_error = total_error
            best_solution = (Iy_id, kf_id)

    results.append({
        "i": i,
        "j": j,

        "Iy_true": Iy_true,
        "kf_true": kf_true,

        "success_count": success_count,
        "success_rate": success_count / num_inits,

        "best_error": best_error,

        "Iy_best": best_solution[0],
        "kf_best": best_solution[1],
    })

    print(
        f"Sample {s+1}/{num_samples} | "
        f"success rate: {success_count}/{num_inits}"
    )

df_results = pd.DataFrame(results)
df_results.to_csv("csv_files\\fem_inversion_success_2D_42.csv", index=False)

print("DONE — Results saved to fem_inversion_success_2D.csv")

## **Three identified modes and two parameters**

In [ ]:
N = param_space2D_fixed.shape[0]
DOF = eigvecs2D_fixed.shape[3]

kf_min, kf_max = 5e6, 5e7
Iy_min, Iy_max = 100, 2000

kf_vals = np.linspace(kf_min, kf_max, N)
Iy_vals = np.linspace(Iy_min, Iy_max, N)

def params_from_indices(i, j):
    return Iy_vals[i], kf_vals[j]

def objective(theta, f_meas, modes_meas):
    Iy, kf = theta

    i = np.argmin(np.abs(Iy_vals - Iy))
    j = np.argmin(np.abs(kf_vals - kf))

    f_pred = param_space2D_fixed[i, j, :]
    modes_pred = eigvecs2D_fixed[i, j, :, :]

    # frequency mismatch
    freq_term = np.sum(((f_pred - f_meas) / f_meas) ** 2)

    # mode shape mismatch
    mode_term = 0.0

    for r in range(3):
        phi_p = modes_pred[:, r]
        phi_m = modes_meas[:, r]

        gamma = (phi_p @ phi_m) / (phi_m @ phi_m)

        mode_term += (
            np.linalg.norm(phi_p - gamma * phi_m) ** 2
            / np.linalg.norm(gamma * phi_m) ** 2
        )

    return freq_term + mode_term

num_samples = 1000
num_inits = 50

i_rand = np.random.randint(0, N, num_samples)
j_rand = np.random.randint(0, N, num_samples)

results = []

tol = 0.10  

for s in range(num_samples):

    i = i_rand[s]
    j = j_rand[s]

    # "measured" data from the parameter space
    f_meas = param_space2D_fixed[i, j, :]
    modes_meas = eigvecs2D_fixed[i, j, :, :]

    Iy_true, kf_true = params_from_indices(i, j)

    success_count = 0
    best_error = np.inf
    best_solution = None

    for _ in range(num_inits):

        Iy0 = np.random.uniform(Iy_min, Iy_max)
        kf0 = np.random.uniform(kf_min, kf_max)

        res = minimize(
            objective,
            np.array([Iy0, kf0]),
            args=(f_meas, modes_meas),
            method="Nelder-Mead",
            options={
                "maxiter": 300,
                "disp": False
            }
        )

        Iy_id, kf_id = res.x

        err_Iy = abs(Iy_id - Iy_true) / Iy_true
        err_kf = abs(kf_id - kf_true) / kf_true

        success = (err_Iy <= tol) and (err_kf <= tol)

        success_count += success

        # track best solution
        total_error = (err_Iy + err_kf) / 2

        if total_error < best_error:
            best_error = total_error
            best_solution = (Iy_id, kf_id)

    results.append({
        "i": i,
        "j": j,

        "Iy_true": Iy_true,
        "kf_true": kf_true,

        "success_count": success_count,
        "success_rate": success_count / num_inits,

        "best_error": best_error,

        "Iy_best": best_solution[0],
        "kf_best": best_solution[1],
    })

    print(
        f"Sample {s+1}/{num_samples} | "
        f"success rate: {success_count}/{num_inits}"
    )
    
df_results = pd.DataFrame(results)
df_results.to_csv("csv_files\\fem_inversion_success_2D_32.csv", index=False)

print("DONE — Results saved to fem_inversion_success_2D.csv")

## **Two identified modes and two parameters**

In [ ]:
N = param_space2D_fixed.shape[0]
DOF = eigvecs2D_fixed.shape[3]

kf_min, kf_max = 5e6, 5e7
Iy_min, Iy_max = 100, 2000

kf_vals = np.linspace(kf_min, kf_max, N)
Iy_vals = np.linspace(Iy_min, Iy_max, N)

def params_from_indices(i, j):
    return Iy_vals[i], kf_vals[j]

def objective(theta, f_meas, modes_meas):
    Iy, kf = theta

    i = np.argmin(np.abs(Iy_vals - Iy))
    j = np.argmin(np.abs(kf_vals - kf))

    f_pred = param_space2D_fixed[i, j, :]
    modes_pred = eigvecs2D_fixed[i, j, :, :]

    # frequency mismatch
    freq_term = np.sum(((f_pred - f_meas) / f_meas) ** 2)

    # mode shape mismatch
    mode_term = 0.0

    for r in range(2):
        phi_p = modes_pred[:, r]
        phi_m = modes_meas[:, r]

        gamma = (phi_p @ phi_m) / (phi_m @ phi_m)

        mode_term += (
            np.linalg.norm(phi_p - gamma * phi_m) ** 2
            / np.linalg.norm(gamma * phi_m) ** 2
        )

    return freq_term + mode_term

num_samples = 1000
num_inits = 50

i_rand = np.random.randint(0, N, num_samples)
j_rand = np.random.randint(0, N, num_samples)

results = []

tol = 0.10 

for s in range(num_samples):

    i = i_rand[s]
    j = j_rand[s]

    # "measured" data from the parameter space
    f_meas = param_space2D_fixed[i, j, :]
    modes_meas = eigvecs2D_fixed[i, j, :, :]

    Iy_true, kf_true = params_from_indices(i, j)

    success_count = 0
    best_error = np.inf
    best_solution = None

    for _ in range(num_inits):

        Iy0 = np.random.uniform(Iy_min, Iy_max)
        kf0 = np.random.uniform(kf_min, kf_max)

        res = minimize(
            objective,
            np.array([Iy0, kf0]),
            args=(f_meas, modes_meas),
            method="Nelder-Mead",
            options={
                "maxiter": 300,
                "disp": False
            }
        )

        Iy_id, kf_id = res.x

        err_Iy = abs(Iy_id - Iy_true) / Iy_true
        err_kf = abs(kf_id - kf_true) / kf_true

        success = (err_Iy <= tol) and (err_kf <= tol)

        success_count += success

        # track best solution
        total_error = (err_Iy + err_kf) / 2

        if total_error < best_error:
            best_error = total_error
            best_solution = (Iy_id, kf_id)

    results.append({
        "i": i,
        "j": j,

        "Iy_true": Iy_true,
        "kf_true": kf_true,

        "success_count": success_count,
        "success_rate": success_count / num_inits,

        "best_error": best_error,

        "Iy_best": best_solution[0],
        "kf_best": best_solution[1],
    })

    print(
        f"Sample {s+1}/{num_samples} | "
        f"success rate: {success_count}/{num_inits}"
    )

df_results = pd.DataFrame(results)
df_results.to_csv("csv_files\\fem_inversion_success_2D_22.csv", index=False)

print("DONE — Results saved to fem_inversion_success_2D.csv")

## **Three parameters used in the second 3D dataset**

## **Using four identified modes and three parameters** 

In [ ]:
N = param_space3D_2_fixed.shape[0]
DOF = eigvecs3D_2_fixed.shape[3]

kf_min, kf_max = 5e6, 5e7
Iy_min, Iy_max = 100, 2000
Iywall_min, Iywall_max = 100, 2000

kf_vals = np.linspace(kf_min, kf_max, N)
Iy_vals = np.linspace(Iy_min, Iy_max, N)
Iywall_vals = np.linspace(Iywall_min, Iywall_max, N)

def params_from_indices(i, j, k):
    return Iy_vals[i], Iywall_vals[j], kf_vals[k]

def objective(theta, f_meas, modes_meas):
    Iy, Iywall, kf = theta

    i = np.argmin(abs(Iy_vals - Iy))
    j = np.argmin(abs(Iywall_vals - Iywall))
    k = np.argmin(abs(kf_vals - kf))

    f_pred = param_space3D_2_fixed[i, j, k, :]
    modes_pred = eigvecs3D_2_fixed[i, j, k, :, :]

    freq_term = np.sum(((f_pred - f_meas) / f_meas) ** 2)

    mode_term = 0
    for r in range(4):
        phi_p = modes_pred[:, r]
        phi_m = modes_meas[:, r]

        gamma = (phi_p @ phi_m) / (phi_m @ phi_m)

        mode_term += np.linalg.norm(phi_p - gamma * phi_m) ** 2 / np.linalg.norm(gamma * phi_m) ** 2

    return freq_term + mode_term

num_samples = 1000
num_inits = 50

i_rand = np.random.randint(0, N, num_samples)
j_rand = np.random.randint(0, N, num_samples)
k_rand = np.random.randint(0, N, num_samples)

results = []

tol = 0.1   

for s in range(num_samples):
    i, j, k = i_rand[s], j_rand[s], k_rand[s]

    f_meas = param_space3D_2_fixed[i, j, k, :]
    modes_meas = eigvecs3D_2_fixed[i, j, k, :, :]

    Iy_true, Iywall_true, kf_true = params_from_indices(i, j, k)

    success_count = 0
    best_error = np.inf
    best_solution = None

    for _ in range(num_inits):

        Iy0 = np.random.uniform(Iy_min, Iy_max)
        Iywall0 = np.random.uniform(Iywall_min, Iywall_max)
        kf0 = np.random.uniform(kf_min, kf_max)

        res = minimize(
            objective,
            np.array([Iy0, Iywall0, kf0]),
            args=(f_meas, modes_meas),
            method="Nelder-Mead",
            options={"maxiter": 300, "disp": False}
        )

        Iy_id, Iywall_id, kf_id = res.x

        err_Iy = abs(Iy_id - Iy_true) / Iy_true
        err_Iywall = abs(Iywall_id - Iywall_true) / Iywall_true
        err_kf = abs(kf_id - kf_true) / kf_true

        success = (err_Iy <= tol) and (err_Iywall <= tol) and (err_kf <= tol)

        success_count += success

        # track best solution
        total_error = (err_Iy + err_Iywall + err_kf) / 3
        if total_error < best_error:
            best_error = total_error
            best_solution = (Iy_id, Iywall_id, kf_id)

    results.append({
        "i": i,
        "j": j,
        "k": k,

        "Iy_true": Iy_true,
        "Iywall_true": Iywall_true,
        "kf_true": kf_true,

        "success_count": success_count,
        "success_rate": success_count / num_inits,

        "best_error": best_error,

        "Iy_best": best_solution[0],
        "Iywall_best": best_solution[1],
        "kf_best": best_solution[2],
    })

    print(f"Sample {s+1}/{num_samples} | success rate: {success_count}/{num_inits}")

df_results = pd.DataFrame(results)
df_results.to_csv("csv_files\\fem_inversion_success_3D_2_43.csv", index=False)

print("DONE — Results saved to fem_inversion_success.csv")

## **Using three identified modes and three parameters** 

In [ ]:
N = param_space3D_2_fixed.shape[0]
DOF = eigvecs3D_2_fixed.shape[3]

kf_min, kf_max = 5e6, 5e7
Iy_min, Iy_max = 100, 2000
Iywall_min, Iywall_max = 100, 2000

kf_vals = np.linspace(kf_min, kf_max, N)
Iy_vals = np.linspace(Iy_min, Iy_max, N)
Iywall_vals = np.linspace(Iywall_min, Iywall_max, N)

def params_from_indices(i, j, k):
    return Iy_vals[i], Iywall_vals[j], kf_vals[k]

def objective(theta, f_meas, modes_meas):
    Iy, Iywall, kf = theta

    i = np.argmin(abs(Iy_vals - Iy))
    j = np.argmin(abs(Iywall_vals - Iywall))
    k = np.argmin(abs(kf_vals - kf))

    f_pred = param_space3D_2_fixed[i, j, k, :]
    modes_pred = eigvecs3D_2_fixed[i, j, k, :, :]

    freq_term = np.sum(((f_pred - f_meas) / f_meas) ** 2)

    mode_term = 0
    for r in range(3):
        phi_p = modes_pred[:, r]
        phi_m = modes_meas[:, r]

        gamma = (phi_p @ phi_m) / (phi_m @ phi_m)

        mode_term += np.linalg.norm(phi_p - gamma * phi_m) ** 2 / np.linalg.norm(gamma * phi_m) ** 2

    return freq_term + mode_term

num_samples = 1000
num_inits = 50

i_rand = np.random.randint(0, N, num_samples)
j_rand = np.random.randint(0, N, num_samples)
k_rand = np.random.randint(0, N, num_samples)

results = []

tol = 0.1   

for s in range(num_samples):
    i, j, k = i_rand[s], j_rand[s], k_rand[s]

    f_meas = param_space3D_2_fixed[i, j, k, :]
    modes_meas = eigvecs3D_2_fixed[i, j, k, :, :]

    Iy_true, Iywall_true, kf_true = params_from_indices(i, j, k)

    success_count = 0
    best_error = np.inf
    best_solution = None

    for _ in range(num_inits):

        Iy0 = np.random.uniform(Iy_min, Iy_max)
        Iywall0 = np.random.uniform(Iywall_min, Iywall_max)
        kf0 = np.random.uniform(kf_min, kf_max)

        res = minimize(
            objective,
            np.array([Iy0, Iywall0, kf0]),
            args=(f_meas, modes_meas),
            method="Nelder-Mead",
            options={"maxiter": 300, "disp": False}
        )

        Iy_id, Iywall_id, kf_id = res.x

        err_Iy = abs(Iy_id - Iy_true) / Iy_true
        err_Iywall = abs(Iywall_id - Iywall_true) / Iywall_true
        err_kf = abs(kf_id - kf_true) / kf_true

        success = (err_Iy <= tol) and (err_Iywall <= tol) and (err_kf <= tol)

        success_count += success

        # track best solution
        total_error = (err_Iy + err_Iywall + err_kf) / 3
        if total_error < best_error:
            best_error = total_error
            best_solution = (Iy_id, Iywall_id, kf_id)

    results.append({
        "i": i,
        "j": j,
        "k": k,

        "Iy_true": Iy_true,
        "Iywall_true": Iywall_true,
        "kf_true": kf_true,

        "success_count": success_count,
        "success_rate": success_count / num_inits,

        "best_error": best_error,

        "Iy_best": best_solution[0],
        "Iywall_best": best_solution[1],
        "kf_best": best_solution[2],
    })

    print(f"Sample {s+1}/{num_samples} | success rate: {success_count}/{num_inits}")

df_results = pd.DataFrame(results)
df_results.to_csv("csv_files\\fem_inversion_success_3D_2_33.csv", index=False)

print("DONE — Results saved to fem_inversion_success.csv")